In [24]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [17]:
df = pd.read_csv('data/covid_patient_dataset.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,67.0,Male,97.2,Strong,Bengaluru,No
1,80.0,Male,102.1,Mild,Surat,Yes
2,NaN,Male,98.6,Mild,Kolkata,No
3,69.0,Female,98.7,Strong,Ahmedabad,No
4,6.0,Female,99.6,Mild,Bengaluru,No


In [11]:
df.isnull().sum()

age          70
gender        0
fever         0
cough         0
city          0
has_covid     0
dtype: int64

In [18]:
df['city'] = df.city.str.lower()
df['cough'] = df['cough'].str.lower()
df['gender'] = df['gender'].str.lower()
df['has_covid'] = df['has_covid'].str.lower()
df['has_covid'] = df['has_covid'].apply(lambda x: 1 if x == 'yes' else 0)
df.head()

,age,gender,fever,cough,city,has_covid
0,67.0,male,97.2,strong,bengaluru,0
1,80.0,male,102.1,mild,surat,1
2,NaN,male,98.6,mild,kolkata,0
3,69.0,female,98.7,strong,ahmedabad,0
4,6.0,female,99.6,mild,bengaluru,0


In [19]:
X = df.drop(columns=['has_covid'])
y = df['has_covid']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [20]:
X_train.head()

,age,gender,fever,cough,city
92,2.0,male,103.8,mild,mumbai
223,37.0,male,100.0,strong,pune
234,NaN,female,97.8,strong,mumbai
232,35.0,female,101.4,mild,jaipur
377,NaN,male,103.0,mild,jaipur


In [21]:
transformer = ColumnTransformer(transformers=[
    ('imputer', SimpleImputer(), ['age']),
    ('ordinal', OrdinalEncoder(dtype=np.int32, categories=[['mild', 'strong']]), ['cough']),
    ('onehot', OneHotEncoder(sparse_output=False, dtype=np.int32, drop = 'first'), ['gender', 'city'])
], 
remainder='passthrough', 
verbose_feature_names_out=False)

In [22]:
new_X_train = pd.DataFrame(transformer.fit_transform(X_train), 
                           index=X_train.index, 
                           columns = transformer.get_feature_names_out())
new_X_test = pd.DataFrame(transformer.transform(X_test), 
                          index=X_test.index, 
                          columns=transformer.get_feature_names_out())

In [23]:
new_X_train.sample(5)

,age,cough,gender_male,city_bengaluru,city_chennai,city_delhi,city_hyderabad,city_jaipur,city_kolkata,city_mumbai,city_pune,city_surat,fever
120,44.57563,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,100.5
369,72.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,98.8
398,44.57563,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,98.7
112,48.00000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,102.9
211,13.00000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,100.1


In [33]:
clf = LogisticRegression(max_iter= 350)

clf.fit(new_X_train, y_train)

y_pred = clf.predict(new_X_test)

accuracy_score(y_test, y_pred)

0.675